# División de Dataset de Imágenes en Train / Valid / Test

Este notebook divide un dataset de imágenes organizado por carpetas de clase en subconjuntos de **train (70%)**, **valid (15%)** y **test (15%)**.

**Estructura de entrada esperada:**
```
Dataset/
    Clase1/
        img1.jpg
        img2.jpg
    Clase2/
        ...
```

**Estructura de salida:**
```
Dataset_split/
    train/
        Clase1/
        Clase2/
    valid/
        Clase1/
        Clase2/
    test/
        Clase1/
        Clase2/
```


## 1. Importar librerías

In [1]:
import random
import shutil
from pathlib import Path

# Extensiones de imagen soportadas
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tiff"}

## 2. Parámetros de configuración

Ajusta estas variables según tu caso antes de correr el resto del notebook.

In [2]:
# Carpeta raíz del dataset original
INPUT_DIR = "Dataset"

# Carpeta donde se guardará el dataset dividido
OUTPUT_DIR = "Dataset_split"

# Proporciones de división (deben sumar 1.0)
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15

# Semilla aleatoria para reproducibilidad
SEED = 42

# Si es True, MUEVE los archivos en lugar de copiarlos
MOVE_FILES = False

## 3. Función principal de división

In [3]:
def split_dataset(input_dir, output_dir,
                   train_ratio=0.70, valid_ratio=0.15, test_ratio=0.15,
                   seed=42, move_files=False):

    assert abs(train_ratio + valid_ratio + test_ratio - 1.0) < 1e-6, \
        "Los ratios deben sumar 1.0"

    input_path = Path(input_dir)
    output_path = Path(output_dir)

    if not input_path.exists():
        raise FileNotFoundError(f"No se encontró la carpeta de entrada: {input_path}")

    random.seed(seed)

    # Detecta las carpetas de clase (subcarpetas dentro de Dataset)
    class_dirs = [d for d in input_path.iterdir() if d.is_dir()]

    if not class_dirs:
        raise ValueError(f"No se encontraron subcarpetas (clases) dentro de {input_path}")

    print(f"Se encontraron {len(class_dirs)} clases: {[d.name for d in class_dirs]}\n")

    resumen = {}

    for class_dir in class_dirs:
        images = [f for f in class_dir.iterdir()
                  if f.is_file() and f.suffix.lower() in IMG_EXTS]

        if not images:
            print(f"⚠️  Advertencia: {class_dir.name} no tiene imágenes válidas, se omite.")
            continue

        random.shuffle(images)

        n_total = len(images)
        n_train = int(n_total * train_ratio)
        n_valid = int(n_total * valid_ratio)
        # El resto va a test (evita perder imágenes por redondeo)
        n_test = n_total - n_train - n_valid

        splits = {
            "train": images[:n_train],
            "valid": images[n_train:n_train + n_valid],
            "test": images[n_train + n_valid:]
        }

        for split_name, split_files in splits.items():
            dest_dir = output_path / split_name / class_dir.name
            dest_dir.mkdir(parents=True, exist_ok=True)

            for img_file in split_files:
                dest_file = dest_dir / img_file.name
                if move_files:
                    shutil.move(str(img_file), str(dest_file))
                else:
                    shutil.copy2(str(img_file), str(dest_file))

        resumen[class_dir.name] = (n_train, n_valid, n_test, n_total)
        print(f"✔ {class_dir.name}: total={n_total} -> train={n_train}, valid={n_valid}, test={n_test}")

    print("\nResumen final:")
    print(f"{'Clase':<20}{'Train':<10}{'Valid':<10}{'Test':<10}{'Total':<10}")
    for clase, (tr, va, te, tot) in resumen.items():
        print(f"{clase:<20}{tr:<10}{va:<10}{te:<10}{tot:<10}")

    print(f"\nListo. Dataset dividido guardado en: {output_path.resolve()}")

    return resumen

## 4. Ejecutar la división

In [4]:
resumen = split_dataset(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    train_ratio=TRAIN_RATIO,
    valid_ratio=VALID_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
    move_files=MOVE_FILES
)

Se encontraron 28 clases: ['Batol_Gourd_Alternaria_Leaf_Blight', 'Batol_Gourd_Anthracnose', 'Batol_Gourd_Downy_Mildew', 'Batol_Gourd_Early_Alternaria_Leaf_Blight', 'Batol_Gourd_Fungal_Damage_Leaf', 'Batol_Gourd_Healthy', 'Batol_Gourd_Mosaic_Virus', 'papaya_Bacterial_Blight', 'papaya_Carica_Insect_Hole', 'papaya_Curled_Yellow_Spot', 'papaya_healthy_leaf', 'papaya_Mosaic_Virus', 'papaya_pathogen_symptoms', 'papaya_Yellow_Necrotic_Spots_Holes', 'Tomato_Downy Mildew', 'Tomato_Healthy', 'Tomato_Mosaic', 'Tomato_Spot', 'Tomato_white_spot', 'Zucchini_Angular_Leaf_Spot', 'Zucchini_Anthracnose', 'Zucchini_Downy_Midew', 'Zucchini_Dry_Leaf', 'Zucchini_Healthy', 'Zucchini_Insect_Damage', 'Zucchini_Iron_Chlorosis_Damage', 'Zucchini_Xanthomonas_Leaf_Spot', 'Zucchini_Yellow_Mosaic_Virus']

✔ Batol_Gourd_Alternaria_Leaf_Blight: total=303 -> train=212, valid=45, test=46
✔ Batol_Gourd_Anthracnose: total=276 -> train=193, valid=41, test=42
✔ Batol_Gourd_Downy_Mildew: total=286 -> train=200, valid=42, tes

## Notas

- Por defecto el script **copia** las imágenes, así que tu carpeta `Dataset` original queda intacta. Cambia `MOVE_FILES = True` si prefieres moverlas en vez de copiarlas.
- La semilla (`SEED`) fija asegura que el split sea reproducible cada vez que corras el notebook.
- Se detectan automáticamente las clases (subcarpetas) y las extensiones de imagen más comunes (jpg, png, bmp, gif, webp, tiff).
